In [54]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyApp") \
    .getOrCreate()

In [55]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql import functions as F
from datetime import datetime


# Datasets

In [56]:
customer_data = [
("C001","Delhi","Premium"),
("C002","Mumbai","Standard"),
("C003","Bangalore","Premium"),
("C004","Chennai","Standard"),
("C005","Mumbai","Premium")
]

In [57]:
sales_data = [
("TXN001","Delhi ","Laptop","Electronics","45000","2024-01-05","Completed"),
("TXN002","Mumbai","Mobile ","electronics","32000","05/01/2024","Completed"),
("TXN003","Bangalore","Tablet"," Electronics ","30000","2024/01/06","Cancelled"),
("TXN004","Delhi","Laptop","Electronics","","2024-01-07","Cancelled"),
("TXN005","Chennai","Mobile","Electronics","invalid","2024-01-08","Completed"),
("TXN006","Mumbai","Tablet","Electronics",None,"2024-01-08","Completed"),
("TXN007","Delhi","Laptop","electronics","45000","09-01-2024","Compled"),
("TXN008","Bangalore","Mobile","Electronics","28000","2024-01-09","Completed"),
("TXN009","Mumbai","Laptop","Electronics","55000","2024-01-10","Completed"),
("TXN009","Mumbai","Laptop","Electronics","55000","2024-01-10","Completed")
]

In [58]:
city_lookup = [
("Delhi","Tier-1"),
("Mumbai","Tier-1"),
("Bangalore","Tier-1"),
("Chennai","Tier-2")
]


# Cleanup

In [59]:
def clean_sales(row):
    txn_id, city, product, category, amount, date_str, status = row

    city = city.strip().title()
    product = product.strip().title()
    category = category.strip().title()

    try:
        amount = int(amount)
    except:
        amount = None
    date_formats = ["%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d", "%d-%m-%Y"]
    txn_date = None
    for fmt in date_formats:
        try:
            txn_date = datetime.strptime(date_str, fmt).date()
            break
        except:
            continue

    status = status.strip().title() if status else "Unknown"

    return (txn_id, city, product, category, amount, txn_date, status)

In [60]:
cleaned_sales = [clean_sales(row) for row in sales_data]

In [61]:
sales_df = sales_df.dropDuplicates()

# Schema

In [62]:
sales_schema = StructType([
    StructField("TransactionID", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Product", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Amount", IntegerType(), True),
    StructField("TransactionDate", DateType(), True),
    StructField("Status", StringType(), True)
])

customer_schema = StructType([
    StructField("CustomerID", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Segment", StringType(), True)
])

city_schema = StructType([
    StructField("City", StringType(), True),
    StructField("Tier", StringType(), True)
])

In [63]:
sales_df = spark.createDataFrame(cleaned_sales, schema=sales_schema)
customer_df = spark.createDataFrame(customer_data, schema=customer_schema)
city_df = spark.createDataFrame(city_lookup, schema=city_schema)

# Show Cleaned Data

In [64]:
sales_df.show()

customer_df.show()

city_df.show()

+-------------+---------+-------+-----------+------+---------------+---------+
|TransactionID|     City|Product|   Category|Amount|TransactionDate|   Status|
+-------------+---------+-------+-----------+------+---------------+---------+
|       TXN001|    Delhi| Laptop|Electronics| 45000|     2024-01-05|Completed|
|       TXN002|   Mumbai| Mobile|Electronics| 32000|     2024-01-05|Completed|
|       TXN003|Bangalore| Tablet|Electronics| 30000|     2024-01-06|Cancelled|
|       TXN004|    Delhi| Laptop|Electronics|  NULL|     2024-01-07|Cancelled|
|       TXN005|  Chennai| Mobile|Electronics|  NULL|     2024-01-08|Completed|
|       TXN006|   Mumbai| Tablet|Electronics|  NULL|     2024-01-08|Completed|
|       TXN007|    Delhi| Laptop|Electronics| 45000|     2024-01-09|  Compled|
|       TXN008|Bangalore| Mobile|Electronics| 28000|     2024-01-09|Completed|
|       TXN009|   Mumbai| Laptop|Electronics| 55000|     2024-01-10|Completed|
|       TXN009|   Mumbai| Laptop|Electronics| 55000|

# Identify corrupt Data

In [65]:
invalid_amount_df = sales_df.filter(F.col("Amount").isNull())


invalid_date_df = sales_df.filter(F.col("TransactionDate").isNull())


duplicate_txn_df = sales_df.groupBy("TransactionID").count().filter(F.col("count") > 1)



In [66]:
invalid_amount_df.show()

invalid_date_df.show()

duplicate_txn_df.show()

+-------------+-------+-------+-----------+------+---------------+---------+
|TransactionID|   City|Product|   Category|Amount|TransactionDate|   Status|
+-------------+-------+-------+-----------+------+---------------+---------+
|       TXN004|  Delhi| Laptop|Electronics|  NULL|     2024-01-07|Cancelled|
|       TXN005|Chennai| Mobile|Electronics|  NULL|     2024-01-08|Completed|
|       TXN006| Mumbai| Tablet|Electronics|  NULL|     2024-01-08|Completed|
+-------------+-------+-------+-----------+------+---------------+---------+

+-------------+----+-------+--------+------+---------------+------+
|TransactionID|City|Product|Category|Amount|TransactionDate|Status|
+-------------+----+-------+--------+------+---------------+------+
+-------------+----+-------+--------+------+---------------+------+

+-------------+-----+
|TransactionID|count|
+-------------+-----+
|       TXN009|    2|
+-------------+-----+



# All null amounts as 0

In [67]:
from pyspark.sql import functions as F

sales_df_clean = sales_df.withColumn(
    "Amount",
    F.when(F.col("Amount").isNull(), F.lit(0)).otherwise(F.col("Amount"))
)

sales_df_clean.show(truncate=False)

+-------------+---------+-------+-----------+------+---------------+---------+
|TransactionID|City     |Product|Category   |Amount|TransactionDate|Status   |
+-------------+---------+-------+-----------+------+---------------+---------+
|TXN001       |Delhi    |Laptop |Electronics|45000 |2024-01-05     |Completed|
|TXN002       |Mumbai   |Mobile |Electronics|32000 |2024-01-05     |Completed|
|TXN003       |Bangalore|Tablet |Electronics|30000 |2024-01-06     |Cancelled|
|TXN004       |Delhi    |Laptop |Electronics|0     |2024-01-07     |Cancelled|
|TXN005       |Chennai  |Mobile |Electronics|0     |2024-01-08     |Completed|
|TXN006       |Mumbai   |Tablet |Electronics|0     |2024-01-08     |Completed|
|TXN007       |Delhi    |Laptop |Electronics|45000 |2024-01-09     |Compled  |
|TXN008       |Bangalore|Mobile |Electronics|28000 |2024-01-09     |Completed|
|TXN009       |Mumbai   |Laptop |Electronics|55000 |2024-01-10     |Completed|
|TXN009       |Mumbai   |Laptop |Electronics|55000 |

# Trim and normalize string columns
# Keep completed orders

In [68]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType
from datetime import datetime

def parse_date(date_str):
    if date_str is None: return None
    formats = ["%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d", "%d-%m-%Y"]
    for fmt in formats:
        try:
            return datetime.strptime(date_str.strip(), fmt).date()
        except:
            continue
    return None

parse_date_udf = F.udf(parse_date, DateType())


In [69]:

sales_df_clean = (
    sales_df_clean
    .withColumn("Category", F.upper(F.trim(F.col("Category"))))
    .withColumn("Status", F.initcap(F.trim(F.col("Status"))))
    .dropDuplicates(["TransactionID", "City", "Product", "TransactionDate"])
    .filter(F.col("Status") == "Completed")
)

sales_df_clean.show(truncate=False)

+-------------+---------+-------+-----------+------+---------------+---------+
|TransactionID|City     |Product|Category   |Amount|TransactionDate|Status   |
+-------------+---------+-------+-----------+------+---------------+---------+
|TXN001       |Delhi    |Laptop |ELECTRONICS|45000 |2024-01-05     |Completed|
|TXN002       |Mumbai   |Mobile |ELECTRONICS|32000 |2024-01-05     |Completed|
|TXN005       |Chennai  |Mobile |ELECTRONICS|0     |2024-01-08     |Completed|
|TXN006       |Mumbai   |Tablet |ELECTRONICS|0     |2024-01-08     |Completed|
|TXN008       |Bangalore|Mobile |ELECTRONICS|28000 |2024-01-09     |Completed|
|TXN009       |Mumbai   |Laptop |ELECTRONICS|55000 |2024-01-10     |Completed|
+-------------+---------+-------+-----------+------+---------------+---------+



# Standard join on City

In [70]:
from pyspark.sql import functions as F


sales_enriched_df = sales_df_clean.join(city_df, on="City", how="left")

sales_enriched_df.show(truncate=False)

+---------+-------------+-------+-----------+------+---------------+---------+------+
|City     |TransactionID|Product|Category   |Amount|TransactionDate|Status   |Tier  |
+---------+-------------+-------+-----------+------+---------------+---------+------+
|Delhi    |TXN001       |Laptop |ELECTRONICS|45000 |2024-01-05     |Completed|Tier-1|
|Mumbai   |TXN002       |Mobile |ELECTRONICS|32000 |2024-01-05     |Completed|Tier-1|
|Chennai  |TXN005       |Mobile |ELECTRONICS|0     |2024-01-08     |Completed|Tier-2|
|Mumbai   |TXN006       |Tablet |ELECTRONICS|0     |2024-01-08     |Completed|Tier-1|
|Bangalore|TXN008       |Mobile |ELECTRONICS|28000 |2024-01-09     |Completed|Tier-1|
|Mumbai   |TXN009       |Laptop |ELECTRONICS|55000 |2024-01-10     |Completed|Tier-1|
+---------+-------------+-------+-----------+------+---------------+---------+------+



# Broadcast

In [71]:
from pyspark.sql.functions import broadcast

sales_enriched_df = sales_df_clean.join(broadcast(city_df), on="City", how="left")

In [72]:
sales_enriched_df.show()

+---------+-------------+-------+-----------+------+---------------+---------+------+
|     City|TransactionID|Product|   Category|Amount|TransactionDate|   Status|  Tier|
+---------+-------------+-------+-----------+------+---------------+---------+------+
|    Delhi|       TXN001| Laptop|ELECTRONICS| 45000|     2024-01-05|Completed|Tier-1|
|   Mumbai|       TXN002| Mobile|ELECTRONICS| 32000|     2024-01-05|Completed|Tier-1|
|  Chennai|       TXN005| Mobile|ELECTRONICS|     0|     2024-01-08|Completed|Tier-2|
|   Mumbai|       TXN006| Tablet|ELECTRONICS|     0|     2024-01-08|Completed|Tier-1|
|Bangalore|       TXN008| Mobile|ELECTRONICS| 28000|     2024-01-09|Completed|Tier-1|
|   Mumbai|       TXN009| Laptop|ELECTRONICS| 55000|     2024-01-10|Completed|Tier-1|
+---------+-------------+-------+-----------+------+---------------+---------+------+



In [73]:
sales_enriched_df.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(LeftOuter, [City])
:- Filter (Status#1115 = Completed)
:  +- Deduplicate [TransactionID#978, City#979, Product#980, TransactionDate#983]
:     +- Project [TransactionID#978, City#979, Product#980, Category#1114, Amount#1091, TransactionDate#983, initcap(trim(Status#984, None)) AS Status#1115]
:        +- Project [TransactionID#978, City#979, Product#980, upper(trim(Category#981, None)) AS Category#1114, Amount#1091, TransactionDate#983, Status#984]
:           +- Project [TransactionID#978, City#979, Product#980, Category#981, CASE WHEN isnull(Amount#982) THEN 0 ELSE Amount#982 END AS Amount#1091, TransactionDate#983, Status#984]
:              +- LogicalRDD [TransactionID#978, City#979, Product#980, Category#981, Amount#982, TransactionDate#983, Status#984], false
+- ResolvedHint (strategy=broadcast)
   +- LogicalRDD [City#988, Tier#989], false

== Analyzed Logical Plan ==
City: string, TransactionID: string, Product: string, Category: string,

# Revenue Per City

In [75]:
from pyspark.sql import functions as F

revenue_per_city = (
    sales_enriched_df
    .groupBy("City")
    .agg(F.sum("Amount").alias("TotalRevenue"))
    .orderBy(F.desc("TotalRevenue"))
)

revenue_per_city.show()

+---------+------------+
|     City|TotalRevenue|
+---------+------------+
|   Mumbai|       87000|
|    Delhi|       45000|
|Bangalore|       28000|
|  Chennai|           0|
+---------+------------+



# Revenue per Product

In [77]:
revenue_per_product = (
    sales_enriched_df
    .groupBy("Product")
    .agg(F.sum("Amount").alias("TotalRevenue"))
    .orderBy(F.desc("TotalRevenue"))
)

revenue_per_product.show()

+-------+------------+
|Product|TotalRevenue|
+-------+------------+
| Laptop|      100000|
| Mobile|       60000|
| Tablet|           0|
+-------+------------+



# Rank Cities by Total Revenue

In [78]:
from pyspark.sql.window import Window

city_window = Window.orderBy(F.desc("TotalRevenue"))

ranked_cities = (
    revenue_per_city
    .withColumn("Rank", F.rank().over(city_window))
)

ranked_cities.show()

+---------+------------+----+
|     City|TotalRevenue|Rank|
+---------+------------+----+
|   Mumbai|       87000|   1|
|    Delhi|       45000|   2|
|Bangalore|       28000|   3|
|  Chennai|           0|   4|
+---------+------------+----+



# Rank Products Within Each City

In [80]:
product_city_window = Window.partitionBy("City").orderBy(F.desc("TotalRevenue"))

revenue_per_product_city = (
    sales_enriched_df
    .groupBy("City", "Product")
    .agg(F.sum("Amount").alias("TotalRevenue"))
    .withColumn("RankWithinCity", F.rank().over(product_city_window))
)

revenue_per_product_city.show()

+---------+-------+------------+--------------+
|     City|Product|TotalRevenue|RankWithinCity|
+---------+-------+------------+--------------+
|Bangalore| Mobile|       28000|             1|
|  Chennai| Mobile|           0|             1|
|    Delhi| Laptop|       45000|             1|
|   Mumbai| Laptop|       55000|             1|
|   Mumbai| Mobile|       32000|             2|
|   Mumbai| Tablet|           0|             3|
+---------+-------+------------+--------------+



# Identify top-performing city per day

In [82]:
city_day_window = Window.partitionBy("TransactionDate").orderBy(F.desc("DailyRevenue"))

top_city_per_day = (
    sales_enriched_df
    .groupBy("TransactionDate", "City")
    .agg(F.sum("Amount").alias("DailyRevenue"))
    .withColumn("Rank", F.rank().over(city_day_window))
    .filter(F.col("Rank") == 1)  )

top_city_per_day.show()

+---------------+---------+------------+----+
|TransactionDate|     City|DailyRevenue|Rank|
+---------------+---------+------------+----+
|     2024-01-05|    Delhi|       45000|   1|
|     2024-01-08|   Mumbai|           0|   1|
|     2024-01-08|  Chennai|           0|   1|
|     2024-01-09|Bangalore|       28000|   1|
|     2024-01-10|   Mumbai|       55000|   1|
+---------------+---------+------------+----+



# Cache

In [83]:
sales_enriched_df.cache()
sales_enriched_df.count()

6

# Repartition Data by City



In [84]:

sales_partitioned_df = sales_enriched_df.repartition("City")

print("Partitions:", sales_partitioned_df.rdd.getNumPartitions())

Partitions: 1


# Why partition helps?

In [85]:
sales_partitioned_df.explain(True)

== Parsed Logical Plan ==
'RepartitionByExpression ['City]
+- Project [City#979, TransactionID#978, Product#980, Category#1114, Amount#1091, TransactionDate#983, Status#1115, Tier#989]
   +- Join LeftOuter, (City#979 = City#988)
      :- Filter (Status#1115 = Completed)
      :  +- Deduplicate [TransactionID#978, City#979, Product#980, TransactionDate#983]
      :     +- Project [TransactionID#978, City#979, Product#980, Category#1114, Amount#1091, TransactionDate#983, initcap(trim(Status#984, None)) AS Status#1115]
      :        +- Project [TransactionID#978, City#979, Product#980, upper(trim(Category#981, None)) AS Category#1114, Amount#1091, TransactionDate#983, Status#984]
      :           +- Project [TransactionID#978, City#979, Product#980, Category#981, CASE WHEN isnull(Amount#982) THEN 0 ELSE Amount#982 END AS Amount#1091, TransactionDate#983, Status#984]
      :              +- LogicalRDD [TransactionID#978, City#979, Product#980, Category#981, Amount#982, TransactionDate#98

# Write data to parquet

In [86]:

sales_enriched_df.write.mode("overwrite").parquet("/content/cleaned_sales_parquet")

# Write Aggregated Data to ORC

In [87]:
revenue_per_city.write.mode("overwrite").orc("/content/revenue_city_orc")

# Compare Sizes

In [88]:
import os

parquet_size = sum(os.path.getsize(os.path.join(dp, f))
                   for dp, dn, filenames in os.walk("/content/cleaned_sales_parquet")
                   for f in filenames)

orc_size = sum(os.path.getsize(os.path.join(dp, f))
               for dp, dn, filenames in os.walk("/content/revenue_city_orc")
               for f in filenames)

print("Parquet size (bytes):", parquet_size)
print("ORC size (bytes):", orc_size)

Parquet size (bytes): 15965
ORC size (bytes): 504


# Spark Structured Streaming with Avro

In [7]:
!pip uninstall -y pyspark
!pip install pyspark==3.5.1

Found existing installation: pyspark 3.5.1
Uninstalling pyspark-3.5.1:
  Successfully uninstalled pyspark-3.5.1
  Using cached pyspark-3.5.1-py2.py3-none-any.whl
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.1 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [1]:

from pyspark.sql import SparkSession
spark=SparkSession.builder\
.appName("AvroStable")\
.config("spark.jars.packages",
        "org.apache.spark:spark-avro_2.12:3.5.1")\
.getOrCreate()

In [2]:
parquet_df = spark.read.parquet("/content/cleaned_sales_parquet")
parquet_df.printSchema()
parquet_df.show(truncate=False)

root
 |-- City: string (nullable = true)
 |-- TransactionID: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Amount: integer (nullable = true)
 |-- TransactionDate: date (nullable = true)
 |-- Status: string (nullable = true)
 |-- Tier: string (nullable = true)

+---------+-------------+-------+-----------+------+---------------+---------+------+
|City     |TransactionID|Product|Category   |Amount|TransactionDate|Status   |Tier  |
+---------+-------------+-------+-----------+------+---------------+---------+------+
|Bangalore|TXN008       |Mobile |ELECTRONICS|28000 |2024-01-09     |Completed|Tier-1|
|Chennai  |TXN005       |Mobile |ELECTRONICS|0     |2024-01-08     |Completed|Tier-2|
|Mumbai   |TXN006       |Tablet |ELECTRONICS|0     |2024-01-08     |Completed|Tier-1|
|Mumbai   |TXN002       |Mobile |ELECTRONICS|32000 |2024-01-05     |Completed|Tier-1|
|Mumbai   |TXN009       |Laptop |ELECTRONICS|55000 |2024-01-10     |Comple

In [3]:

parquet_df.write.format("avro").mode("overwrite").save("/content/avro_out")